In [ ]:
import os
import openreview

from dotenv import load_dotenv
load_dotenv()
from openreview.api import OpenReviewClient as ClientV2
from openreview import Client as ClientV1

def make_clients() -> tuple[ClientV1, ClientV2]:
    user = os.environ["OPENREVIEW_LOGIN"]
    pwd = os.environ["OPENREVIEW_MDP"]
    v1 = ClientV1(
        baseurl="https://api.openreview.net",
        username=user, password=pwd,
    )
    v2 = ClientV2(
        baseurl="https://api2.openreview.net",
        username=user, password=pwd,
    )
    return v1, v2

V1, V2 = make_clients()

def get_forum_id_from_review(review_id: str) -> str | None:
    for client in (V1, V2):
        try:
            note = client.get_note(review_id)
            if note and note.forum:
                return note.forum
        except openreview.OpenReviewException:
            continue
    return None

In [ ]:
def fetch_pdf_from_review_id(
    review_id: str,
    output_dir: str = os.path.join(os.getenv("CACHE_DIR", "."), "papers_pdf"),
) -> str | None:
    forum_id = get_forum_id_from_review(review_id)
    if not forum_id:
        return None

    os.makedirs(output_dir, exist_ok=True)
    file_path = os.path.join(output_dir, f"{forum_id}.pdf")
    if os.path.exists(file_path):
        return file_path

    for client in (V2, V1):  # V2 d'abord, plus de papiers récents
        try:
            pdf_bytes = client.get_pdf(id=forum_id)
            with open(file_path, "wb") as f:
                f.write(pdf_bytes)
            return file_path
        except openreview.OpenReviewException:
            continue
    return None

In [ ]:
fetch_pdf_from_review_id("B1eTCTj9n7")

In [ ]:
from hydra import initialize, compose
from omegaconf import DictConfig
import os

from src.data.dataloader import data_loader
from src.data.dataset_aprdb import AprdbDataset

import src.config

with initialize(version_base=None, config_path="../conf"):
    cfg: DictConfig = compose(config_name="config",overrides=["+experiment/dataset/aprbd@dataset=aprdb_tt_test_all","experiment/data_loader=1_4_true"])

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

print(cfg)

In [ ]:
ds_test = AprdbDataset(config=cfg.dataset)
len(ds_test)

In [ ]:
import pandas as pd
rows = []
for i in range(len(ds_test)):
    item = ds_test[i]
    rows.append(item)
df = pd.DataFrame(rows)

In [ ]:
df.columns

In [ ]:
df_hwt_only = df[df['label'] == 0]
len(df_hwt_only)

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

CHECKPOINT = Path(os.getenv("CACHE_DIR", ".")) / "paper_sources.parquet"

def load_checkpoint() -> set[str]:
    if CHECKPOINT.exists():
        return set(pd.read_parquet(CHECKPOINT)["review_id"])
    return set()

def append_checkpoint(rows: list[dict]) -> None:
    df_new = pd.DataFrame(rows)
    if CHECKPOINT.exists():
        df_new = pd.concat([pd.read_parquet(CHECKPOINT), df_new], ignore_index=True)
    df_new.to_parquet(CHECKPOINT, index=False)

done = load_checkpoint()
buffer, FLUSH_EVERY = [], 50

for _, row in tqdm(df_hwt_only.iterrows(), total=len(df_hwt_only)):
    rid = row["review-id"]
    if rid in done:
        continue
    try:
        path = fetch_pdf_from_review_id(rid)
    except Exception as e:
        path = None
        tqdm.write(f"[ERR] {rid}: {type(e).__name__}: {e}")

    buffer.append({
        "review_id": rid, "pdf_path": path,
        "paper_id": row["PaperId"], "year": row["year"],
        "conference": row["conference"],
    })
    if len(buffer) >= FLUSH_EVERY:
        append_checkpoint(buffer); buffer.clear()

if buffer:
    append_checkpoint(buffer)

paper_source_paths = pd.read_parquet(CHECKPOINT)

In [ ]:
paper_source_paths

In [ ]:
valid_review = paper_source_paths[paper_source_paths['pdf_path'].notna()]
valid_review.columns

In [ ]:
import pandas as pd

df_clean = valid_review
df_clean = df_clean.copy()

# 1. Création de la clé composite sur les deux DataFrames
df_clean["PaperIdFull"] = (
    df_clean["conference"].astype(str) + "-"
    + df_clean["year"].astype(str) + "-"
    + df_clean["paper_id"].astype(str)
)

df["PaperIdFull"] = (
    df["conference"].astype(str) + "-"
    + df["year"].astype(str) + "-"
    + df["PaperId"].astype(str)
)

# 2. On ne garde de df_clean que la clé + les colonnes à propager
#    (drop_duplicates au cas où plusieurs reviews du même papier seraient dans df_clean)
mapping = (
    df_clean[["PaperIdFull", "paper_id", "pdf_path"]]
    .drop_duplicates(subset="PaperIdFull")
)


# Garde-fou : vérifie la dépendance fonctionnelle avant merge
assert df_clean.groupby("PaperIdFull")[["paper_id", "pdf_path"]].nunique().le(1).all().all(), \
    "Violation : un PaperIdFull est associé à plusieurs paper_id/pdf_path dans df_clean"

mapping = df_clean[["PaperIdFull", "paper_id", "pdf_path"]].drop_duplicates()
df_final = df.merge(mapping, on="PaperIdFull", how="inner")

In [ ]:
df_final.dropna(subset=["pdf_path"], inplace=True)
df_final